In [1]:
import torch
from torch.utils.data import DataLoader, Subset
from transformers import T5TokenizerFast, CLIPProcessor, CLIPTokenizerFast, CLIPImageProcessorFast, T5ForConditionalGeneration


from peft import LoraConfig, get_peft_model, TaskType
from torch.amp import autocast, GradScaler
from tqdm import tqdm
import os 
import json

In [2]:
from Modules.config import (TRAIN_IMAGE_DIR,
                            TEST_IMAGE_DIR,
                            FAISS_IMAGE_PATH,
                            TRAIN_METADATA_PATH,
                            TEST_METADATA_PATH,
                            CLIP_MODEL_NAME,
                            T5_MODEL_NAME,
                            VLM_CHECKPOINT_DIR,
                            T5_DECODER_LORA_CONFIG)

from Modules.FusionVLM import FusionVLM, create_default_FusionVLM, load_default_FusionVLM, save_FusionVLM, apply_lora_config, freeze_encoders
from Modules.retrieval_module import Retriever
from Modules.datasets import VLMDataset, VLMDataCollator
from Modules.utils import print_model_param_stats, add_dict
from Modules.metrics import evaluate_captioning, setup_nltk
from Modules.train_VLM import train_and_evaluate_model

In [3]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE

'cuda'

In [4]:
CLIP_processor = CLIPImageProcessorFast.from_pretrained(CLIP_MODEL_NAME, local_files_only=True)
CLIP_tokenizer = CLIPTokenizerFast.from_pretrained(CLIP_MODEL_NAME, local_files_only=True)
T5_tokenizer = T5TokenizerFast.from_pretrained(T5_MODEL_NAME, local_files_only=True)

In [5]:
collator_T5 = VLMDataCollator(CLIP_processor, T5_tokenizer, device=DEVICE)
collator_CLIP = VLMDataCollator(CLIP_processor, CLIP_tokenizer, label_tokenizer=T5_tokenizer, max_seq_len=77, device=DEVICE)

collator = collator_T5
# collator = collator_CLIP

In [6]:
retriever = Retriever(metadata_path=TRAIN_METADATA_PATH, faiss_path=FAISS_IMAGE_PATH)

train_dataset = VLMDataset(image_dir=TRAIN_IMAGE_DIR, 
                           ref_image_dir=TRAIN_IMAGE_DIR,
                           metadata_path=TRAIN_METADATA_PATH,
                           retriever=retriever)

test_dataset = VLMDataset(image_dir=TEST_IMAGE_DIR, 
                           ref_image_dir=TRAIN_IMAGE_DIR,
                           metadata_path=TEST_METADATA_PATH,
                           retriever=retriever)

In [7]:
BATCH_SIZE = 16
NUM_WORKERS = 0

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, collate_fn=collator)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, collate_fn=collator)

In [8]:
# import numpy as np
# from torch.utils.data import DataLoader, Subset

# num_train_samples = 128
# num_test_samples = 128


# indices = np.random.choice(len(train_dataset), num_train_samples, replace=False)
# train_subset = Subset(train_dataset, indices)
# train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, collate_fn=collator)

# # # indices = np.random.choice(len(test_dataset), num_test_samples, replace=False)
# # # test_subset = Subset(test_dataset, indices)
# # # test_loader = DataLoader(test_subset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, collate_fn=collator)

In [9]:
model = create_default_FusionVLM().to(DEVICE)

c:\Users\Mahan\Documents\Projects\Retrieval-Augmented-Image-Captioning\.venv\Lib\site-packages\peft\tuners\tuners_utils.py:1225: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)


In [10]:
print_model_param_stats(model)

Module                                          Total    Trainable       Frozen
--------------------------------------------------------------------------------
vision_encoder                             87,456,000            0   87,456,000
text_encoder                              109,628,544            0  109,628,544
vision_proj                                   590,592      590,592            0
text_proj                                     590,592      590,592            0
fusion_blocks                              56,724,480   56,724,480            0
post_fusion_ln                                  1,536        1,536            0
fusion_proj                                   590,592      590,592            0
text_decoder                              254,655,744   31,752,192  222,903,552
--------------------------------------------------------------------------------
TOTAL                                     510,238,080   90,249,984  419,988,096


In [11]:
NUM_EPOCHS = 10
full_history = {}
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4,
    weight_decay=0.01
)

setup_nltk()
# os.makedirs(VLM_CHECKPOINT_DIR, exist_ok=True)

In [ ]:
history = train_and_evaluate_model(model, train_loader, optimizer, NUM_EPOCHS, test_loader, T5_tokenizer)
add_dict(full_history, history)

Epoch 1:   0%|          | 0/1924 [00:00<?, ?it/s]

In [ ]:
with open('history0-10.json', "w", encoding="utf-8") as f:
    json.dump(full_history, f, indent=2, ensure_ascii=False)

In [ ]:
NUM_EPOCHS = 5
history = train_and_evaluate_model(model, train_loader, optimizer, NUM_EPOCHS, test_loader, T5_tokenizer, save_interval=1)
add_dict(full_history, history)

In [ ]:
hist = {}
for key in full_history.keys():
    hist[key] = full_history[key][0] + full_history[key][1]

In [ ]:
with open('history0-15.json', "w", encoding="utf-8") as f:
    json.dump(hist, f, indent=2, ensure_ascii=False)

### Exepriments

In [ ]:
model.eval()
loaded.eval()

model.cpu()
loaded.cpu()

In [ ]:
def same_architecture(model_a, model_b):
    keys_a = list(model_a.state_dict().keys())
    keys_b = list(model_b.state_dict().keys())
    return keys_a == keys_b

print("Same architecture:", same_architecture(model, loaded))

In [ ]:
def same_shapes(model_a, model_b):
    sd_a = model_a.state_dict()
    sd_b = model_b.state_dict()

    for k in sd_a:
        if sd_a[k].shape != sd_b[k].shape:
            print(f"Shape mismatch at {k}: {sd_a[k].shape} vs {sd_b[k].shape}")
            return False
    return True

print("Same tensor_shape:", same_shapes(model, loaded))


In [ ]:
def same_weights_exact(model_a, model_b):
    for (ka, va), (kb, vb) in zip(
        model_a.state_dict().items(),
        model_b.state_dict().items()
    ):
        if not torch.equal(va, vb):
            print(f"Mismatch at {ka}")
            return False
    return True

print("Exact same weights:", same_weights_exact(model, loaded))

In [ ]:
with open('history0-15.json', "r", encoding="utf-8") as f:
    hist = json.load(f)

In [ ]:
import matplotlib.pyplot as plt

# for key in hist.keys():
#     plt.plot(hist[key])
#     plt.title(key)
#     plt.show()

plt.figure(figsize=(10, 6))

for key, values in hist.items():
    # if key != 'train_loss' and key != 'val_loss':
        plt.plot(values, label=key)

plt.title("Training History")
plt.xlabel("Step")
plt.ylabel("Value")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
model_orig.to(DEVICE)
model_loaded.to(DEVICE)

In [ ]:
with torch.no_grad():
    for batch in test_loader:
        gt_captions = batch["all_captions"]  # List[List[str]]

        generated_ids_o = model_orig.generate(
            query_pixel_values=batch["query_pixel_values"],
            retrieved_pixel_values=batch.get("retrieved_pixel_valuess"),
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            max_length=None,
            max_new_tokens=64,
            # do_sample=True,
            # temperature=0.9,
            # top_p=0.9,
            # num_beams=4,
            early_stopping=True,
            length_penalty=1.2, # longer outputs 
            repetition_penalty=1.2 # penalize repeats
        )
        
        generated_ids_l = model_loaded.generate(
            query_pixel_values=batch["query_pixel_values"],
            retrieved_pixel_values=batch.get("retrieved_pixel_valuess"),
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            max_length=None,
            max_new_tokens=64,
            # do_sample=True,
            # temperature=0.9,
            # top_p=0.9,
            # num_beams=4,
            early_stopping=True,
            length_penalty=1.2, # longer outputs 
            repetition_penalty=1.2 # penalize repeats
        )

        decoded_o = T5_tokenizer.batch_decode(generated_ids_o, skip_special_tokens=True)
        decoded_l = T5_tokenizer.batch_decode(generated_ids_l, skip_special_tokens=True)
        for i in range(len(decoded_o)):
            print(gt_captions[i][0])
            print(decoded_o[i])   
            print(decoded_l[i])   
            print()         
        break

In [ ]:
with torch.no_grad():
    for batch in test_loader:
        gt_captions = batch["all_captions"]  # List[List[str]]

        generated_ids = model_loaded.generate(
            query_pixel_values=batch["query_pixel_values"],
            retrieved_pixel_values=batch.get("retrieved_pixel_valuess"),
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            max_length=None,
            max_new_tokens=64,
            do_sample=True,
            temperature=0.9,
            top_p=0.9,
            num_beams=4,
            early_stopping=True,
            length_penalty=1.2, # longer outputs 
            repetition_penalty=1.2 # penalize repeats
        )

        decoded = T5_tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
        for i in range(len(decoded)):
            print(gt_captions[i][0])
            print(decoded[i])   
            print()         
        break

In [ ]:
setup_nltk()

In [ ]:
def evaluate_model(model, loader, tokenizer):
    model.eval()

    preds = []
    refs = []

    with torch.no_grad():
        for batch in loader:
            gt_captions = batch.get("all_captions")  # List[List[str]]
            
            generated_ids = model.generate(
                    query_pixel_values=batch.get("query_pixel_values"),
                    retrieved_pixel_values=batch.get("retrieved_pixel_values"),
                    input_ids=batch.get("input_ids"),
                    attention_mask=batch.get("attention_mask"),
                    max_length=None,
                    max_new_tokens=64,
                    # do_sample=True,
                    # temperature=1.0,
                    # top_p=0.9,
                    # num_beams=4,
                    # early_stopping=True,
                    # length_penalty=1.2, # longer outputs 
                    # repetition_penalty=1.2 # penalize repeats
            )

            decoded = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
            preds.extend(decoded)
            refs.extend(gt_captions)
            # refs.extend([[max(caption, key=len)] for caption in gt_captions])
            
    return evaluate_captioning(preds, refs)

In [ ]:
metrics = evaluate_model(model_10, test_loader, T5_tokenizer)

In [ ]:
for k, v in metrics.items():
    print(f"{k}: {v:.4f}")

In [ ]:
model_10 = load_default_FusionVLM('epoch5')
model_10 = model_10.to(DEVICE)